In [ ]:
#updated on 2023-11-02 08:34

In [ ]:
%autosave 300

In [ ]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.optim import lr_scheduler 
import torch.nn.functional as F
import numpy as np 

import torchvision 

from torchvision import datasets, models, transforms
from torch.cuda.amp import autocast, GradScaler 

import time 
import os 

In [ ]:
from collections import Counter,OrderedDict

In [ ]:
#以下为项目参数设置
weights_file_path = r'E:\KerasPy\pretrained_weight'
project_path=r'.'#项目文件目录 
data_dir = r'/home/jason/data/NWPU45_dataset/20%-Train-ratio-A'

In [ ]:
####################################
#Model Choices
model_selection = 'Ensemble'
#study_target
study_target ="Searching optimal ensemble\n"
####################################

In [ ]:
#以下为定制训练超参数设置 
train_original_size = 256

#以下为共享训练超参数设置 
#num_workers --------------------------
loader_workers = 3
train_Bsize = 64
val_Bsize = 96
#----------------------
pin_memory_bool=True#pin_memory
use_amp = True#Automatic Mixed Precision
#####optiizer_settings
optimizer_ft = None
#-----------------------------

In [ ]:
using_resize_setting = True

if 'NWPU' in data_dir:
    using_resize_setting = False
#---------------------------------------

In [ ]:
def get_weight_dict(weights_dir):
    return {file.split('_')[1]:file for file in os.listdir(weights_dir) \
             if os.path.splitext(file)[-1] == '.pth'} 

In [ ]:
#--------------------------------------------------------------------------
dualCM_b0_wpath = os.path.join('.','weights','dualCM-b0','NWPU45')

dualCM_b0_weights_dict = get_weight_dict(dualCM_b0_wpath)
dualCM_b0_weights_dict

In [ ]:
#--------------------------------------------------------------------------
singleCM_b0_wpath = os.path.join('.','weights','singleCM-b0','NWPU45')

singleCM_b0_weights_dict = get_weight_dict(singleCM_b0_wpath)
singleCM_b0_weights_dict

In [ ]:
#--------------------------------------------------------------------------
Rand_b3_wpath = os.path.join('.','weights','rand-b3','NWPU45')
Rand_b3_weights_dict = get_weight_dict(Rand_b3_wpath)
Rand_b3_weights_dict
#

In [ ]:
#--------------------------------------------------------------------------
sorted_b3_wpath = os.path.join('.','weights','sorted_b3','NWPU45')
sorted_b3_weights_dict = get_weight_dict(sorted_b3_wpath)
sorted_b3_weights_dict
#

In [ ]:
#N-VIT
NViT_wpath = os.path.join('.','weights','NViT','NWPU45')
NViT_weights_dict = get_weight_dict(NViT_wpath)
NViT_weights_dict
#

In [ ]:
#Swin-T
Swin_T_wpath = os.path.join('.','weights','Swin-T','NWPU45')
Swin_T_weights_dict = get_weight_dict(Swin_T_wpath)
Swin_T_weights_dict
#

In [ ]:
#R50
R50_wpath = os.path.join('.','weights','R50','NWPU45')
R50_weights_dict = get_weight_dict(R50_wpath)
R50_weights_dict
#

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
def get_training_DA_strategy_A(resize=(True,256),cCrop=(False,256), HFlip =True,VFlip=True,Rotation=True,CJitter=False, RErase=False,
                            ):
    transforms_Compose_list = []
    if resize[0]:
        transforms_Compose_list.append (transforms.Resize(size=resize[1], interpolation = transforms.InterpolationMode.BILINEAR, 
                                                      ))
    if cCrop[0]:
        transforms_Compose_list.append(transforms.CenterCrop(cCrop[1]))
    if CJitter:
        transforms_Compose_list.append(transforms.RandomApply(torch.nn.ModuleList([transforms.ColorJitter(0.5,0.5,0.5)]), 
                                                                                   p = ColorJitter_prob_settings)) 
    if HFlip:
        transforms_Compose_list.append(transforms.RandomHorizontalFlip(p = RHF_prob))
    if VFlip:
        transforms_Compose_list.append(transforms.RandomVerticalFlip(p = RVF_prob))
    if Rotation:
        transforms_Compose_list.append(transforms.RandomApply(torch.nn.ModuleList([transforms.RandomRotation(degrees=180, 
            interpolation = transforms.InterpolationMode.BILINEAR),]),p = Rotation_prob_settings))     
        
    
    transforms_Compose_list.append(transforms.ToTensor())
    transforms_Compose_list.append(transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))
    if RErase:
        transforms_Compose_list.append(transforms.RandomErasing(p = RErase_prob_setting,inplace =True))
    
    return transforms_Compose_list

def get_testing_DA_strategy_A(resize=(True,256),cCrop=(False,256)):    
    transforms_Compose_list = []

    if resize[0]:
        transforms_Compose_list.append (transforms.Resize(size = resize[1], interpolation = transforms.InterpolationMode.BILINEAR, 
                                                      )) 
    if cCrop[0]:
        transforms_Compose_list.append(transforms.CenterCrop(cCrop[1])) 
    transforms_Compose_list.append(transforms.ToTensor())
    
    transforms_Compose_list.append(transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]))
    
    return transforms_Compose_list

In [ ]:
def my_save_txt(filepathname, context):
    number_lines = context.matrix.shape[0]

    matrix_list = []
    for i in range(number_lines):
        current_value_line = context.matrix[i].cpu().tolist()
        matrix_list.append(current_value_line)
        
    with open (filepathname,'w+') as fw:
        fw.writelines(str(matrix_list))
        fw.close()  

In [ ]:
@torch.inference_mode()
def train_model(model_A, model_B, num_epochs = 99):
    global dataset_loader_batch_size
    
    torch.cuda.empty_cache()
   
    model_save_path = os.path.join(project_path,model_sav_pth_filename)
   
    
    logfile = training_logfile_name#train log files 训练记录
    logfile_path = os.path.join(project_path,training_logfile_name)
    
    valing_dataloading_times_per_epoch = (dataset_sizes['val']//val_Bsize)+1
    
    since = time.time()
    
    with open(logfile_path,'a+') as f:
        f.write(str(time.ctime())+'\n'+ f'study_target = {study_target}\n')
        f.write(f'model_selection = {model_selection}\n')
        f.write(f'model_A_selection = {model_A_selection} \n')
        f.write(f'model_B_selection = {model_B_selection} \n')
        f.write(f'training data_dir={data_dir} \n')
        f.write(str(image_datasets['val'])+ '\n')
        f.write('-' * 10+'start training'+'-' * 10+'\n')  
      
    best_acc = 0.0
    best_acc_epoch=0
    
    model_A.eval()
    model_B.eval()
    
    for epoch in range(1, num_epochs+1):
        balance_factor = 0.01 * epoch
        print('Epoch {}/{}'.format(epoch, num_epochs ))
        print('-' * 10)        
        with open(logfile_path,'a+') as f:
            f.write('Epoch {}/{}'.format(epoch, num_epochs )+'\n')
            #f.write('-' * 10+'\n')

        # Each epoch has a training and validation phase
        
        for phase in ['val']:
                
            loss_tmp= float(0.0)
            inputs_size=float(0.0)
            running_loss = float(0.0)
            running_corrects = 0
            static_progress=0 
            show_status_mark=0
            samples_skipped=0
            #
            iter_times_epoch=0
            
            # Iterate over data. ..........................
            print('{:.2f}%>>'.format(0),end='')
            for i,(inputs, labels) in enumerate(dataloaders[phase]):
                torch.cuda.empty_cache()
                inputs_size=labels.size(0)
                preds = None
                loss = None                
               
                inputs.requires_grad=False
                inputs = inputs.to(device)
                labels.requires_grad=False
                labels = labels.to(device)

                #显示进度百分比开始...............
                
                show_status_mark+=1
                static_progress+=1
                if phase =='train':
                    loading_times=training_dataloading_times_per_epoch
                else:
                    loading_times=valing_dataloading_times_per_epoch                    
                
                if  show_status_mark>20:
                    show_status_mark=0
                    print('{:.1f}%>>'.format(static_progress*100/loading_times),end='')
                    #显示进度百分比结束................
                    
                with torch.set_grad_enabled(False):
                    outputs_A = model_A(inputs)
                    outputs_B = model_B(inputs)
                    
                    outputs = outputs_A* balance_factor  + outputs_B * (1 - balance_factor)
                        
                    _, preds = torch.max(outputs, 1)
                    
                running_corrects += torch.sum(preds == labels.data)
                    
            #Iterate process over here......................

            epoch_acc = running_corrects.double() / dataset_sizes[phase]
 
            print('{} Acc: {:.4f} balance_factor: {} Best_Acc: {} '.format(
                phase, epoch_acc, balance_factor, best_acc))
            with open(logfile_path,'a+') as f:
                f.write('{}Acc: {:.4f} BestAcc: {:.4f} balance_factor: {}'.format(
                phase, epoch_acc, best_acc,balance_factor)+'\n')
                if phase == 'val':
                    f.write(str(time.ctime())+'\n')
                    f.write('-' * 10+'\n')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_acc_epoch = epoch
       
    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:4f} '.format(best_acc))
    
    with open(logfile_path,'a+') as f:
        f.write('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60)+'\n')
        f.write('Best val Acc: {:4f}'.format(best_acc)+'\n')
        f.write(f'Best val Acc at epoch={best_acc_epoch} \n')

        f.write('#'*30+'\n')
        f.write('\n')    

    
    return True

In [ ]:
def get_b0_model(wpath:str,wdict:dict):
    
    weights_file = torch.load(os.path.join(wpath, wdict[TR_flag]))
    #------------------------------------------------   
    model_cur = models.efficientnet_b0(weights = None)    
    model_cur.classifier[1] = nn.Linear(in_features = model_cur.classifier[1].in_features,\
                            out_features=class_num,bias=True)    
    print(f'{wdict[TR_flag]} {model_cur.load_state_dict(weights_file)}')
    return model_cur

In [ ]:
def get_b3_model(wpath:str,wdict:dict):
    
    weights_file = torch.load(os.path.join(wpath, wdict[TR_flag]))
    #------------------------------------------------   
    model_cur = models.efficientnet_b3(weights = None)    
    model_cur.classifier[1] = nn.Linear(in_features = model_cur.classifier[1].in_features,\
                            out_features=class_num,bias=True)    
    print(f'{wdict[TR_flag]} {model_cur.load_state_dict(weights_file)}')
    return model_cur

In [ ]:
def get_R50_model(wpath:str,wdict:dict):
    
    weights_file = torch.load(os.path.join(wpath, wdict[TR_flag]))
    #------------------------------------------------   
    model_cur = models.resnet50(weights = None)    
    model_cur.fc = nn.Linear(in_features = model_cur.fc.in_features,\
                            out_features=class_num,bias=True)    
    print(f'{wdict[TR_flag]} {model_cur.load_state_dict(weights_file)}')
    return model_cur

In [ ]:
def get_NviT_model(wpath:str,wdict:dict):
    
    weights_file = torch.load(os.path.join(wpath, wdict[TR_flag]))
    #------------------------------------------------   
    from NextViT.nextvit import nextvit_small
    model_cur = nextvit_small()    
    model_cur.proj_head[0] = nn.Linear(in_features = model_cur.proj_head[0].in_features, \
                            out_features=class_num, bias=True)    
    print(f'{wdict[TR_flag]} {model_cur.load_state_dict(weights_file)}')
    return model_cur

In [ ]:
def get_Swin_T_model(wpath:str,wdict:dict):
    
    weights_file = torch.load(os.path.join(wpath, wdict[TR_flag]))
    #------------------------------------------------   
    model_cur = models.swin_t()    
    model_cur.head = nn.Linear(in_features = model_cur.head.in_features, \
                            out_features=class_num, bias=True)    
    print(f'{wdict[TR_flag]} {model_cur.load_state_dict(weights_file)}')
    return model_cur

In [ ]:
def get_individual_classifier(model_selection):
    if model_selection == 'NViT':
        model_return = get_NviT_model(NViT_wpath, NViT_weights_dict).to(device)
    elif model_selection == 'Swin_T':
        model_return = get_Swin_T_model(Swin_T_wpath, Swin_T_weights_dict).to(device)
    elif model_selection == 'sorted_b3':
        model_return = get_b3_model(sorted_b3_wpath, sorted_b3_weights_dict).to(device)
    elif model_selection == 'rand_b3':
        model_return = get_b3_model(Rand_b3_wpath, Rand_b3_weights_dict).to(device)
    elif model_selection == 'singleCM_b0':
        model_return = get_b0_model(singleCM_b0_wpath, singleCM_b0_weights_dict).to(device)
    elif model_selection == 'dualCM_b0':
        model_return = get_b0_model(dualCM_b0_wpath, dualCM_b0_weights_dict).to(device)
    elif model_selection == 'R50':
        model_return = get_R50_model(R50_wpath, R50_weights_dict).to(device)
    
    return model_return

In [ ]:
def get_modelA(model_a_selection):
    global model_A   
    model_A = get_individual_classifier(model_a_selection)

In [ ]:
def get_modelB(model_b_selection):
    global model_B
    model_B = get_individual_classifier(model_b_selection)

In [ ]:
def get_transform():
    using_resize_setting = True
    if 'NWPU' in data_dir:
        using_resize_setting = False
    if 'UCM21' in data_dir:
        using_cCrop = True
    else: 
        using_cCrop = False
        
    transforms_Compose_list_Testing_256 = \
    get_testing_DA_strategy_A(resize=(using_resize_setting,train_original_size),cCrop=(using_cCrop,256))

    #---------------------------------------------------------------------------------------------------
    global data_transforms
    data_transforms = {
        'val': 
        transforms.Compose(transforms_Compose_list_Testing_256)
    }

In [ ]:
def get_dataset():
    global data_dir
    global image_datasets
    global class_num
    class_num = len([subdir for subdir in os.listdir(os.path.join(data_dir,'train'))])
    image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),data_transforms[x]) for x in [ 'val']}
    global dataset_sizes
    dataset_sizes = {x: len(image_datasets[x]) for x in [ 'val']}
    global class_names
    class_names = image_datasets['val'].classes

In [ ]:
def get_dataloader():
    global dataloaders 
    dataloaders = {}
    dataloaders['val'] =  torch.utils.data.DataLoader(image_datasets['val'], batch_size = val_Bsize,\
                shuffle=True, num_workers=loader_workers,pin_memory=pin_memory_bool, persistent_workers = True) 

In [ ]:
def get_traininglog():
    import platform
    import tools.my_os_utilities as mytools
    global TR_flag
    if platform.system() != 'Windows':
        dataset_flag = data_dir.split('/')[-2].split('_')[0]
        TR_flag = 'TR'+ data_dir.split('/')[-1].split('-')[0] + data_dir[-1]
    else:
        dataset_flag = data_dir.split('\\')[-2].split('_')[0]
        TR_flag = 'TR'+ data_dir.split('\\')[-1].split('-')[0] + data_dir[-1]
    #--------------------------------------------------
    time_stamp = mytools.get_time_prefix('second')
    dataset_log_str = \
    f'{dataset_flag}_{TR_flag}_{time_stamp}_{model_selection}_{model_A_selection}&{model_B_selection}'
    global training_logfile_name
    training_logfile_name = f'Log_{dataset_log_str}.txt'#训练log文件名
    global model_sav_pth_filename
    model_sav_pth_filename = f'{dataset_log_str}.pth'#模型pth文件名

In [ ]:
def initial_training():
    get_transform()
    get_dataset()
    get_dataloader()
    get_traininglog()
    get_modelA(model_A_selection)
    get_modelB(model_B_selection)

In [ ]:
def run_one_cycle():
    for subset in dataset_para_list:
        global data_dir
        data_dir = subset
        initial_training()
        results = train_model( model_A, model_B,num_epochs = 99)

In [ ]:
dataset_para_list = [r'/home/jason/data/NWPU45_dataset/20%-Train-ratio-A',
                     ]

In [ ]:
#1
model_A_selection = 'rand_b3'
model_B_selection = 'NViT'

#------------------------------
run_one_cycle()

In [ ]:
#torch.__version__

In [ ]:
#torchvision.__version__

In [ ]:
%autosave 10

In [ ]:
os.system('shutdown +1')